# M2 - Multimodal RAG Embedding Extraction (Kaggle Cloud)

1. Create a **New Notebook** on Kaggle.
2. Add the **H&M Personalized Fashion Recommendations** dataset to the notebook.
3. Go to Session Options -> Accelerator -> **Turn on GPU T4x2**.
4. Upload this file via File -> Import Notebook and Run All.

## 0. Install Dependencies
Run this cell FIRST and wait for it to complete before running anything else.

In [ ]:
!pip install -q faiss-cpu open_clip_torch
print('Installation complete!')

## 1. Import Libraries & Setup Model

In [ ]:
import os
import pandas as pd
import torch
from PIL import Image
import open_clip
import faiss
import numpy as np
from tqdm.auto import tqdm

DATA_DIR = '../input/h-and-m-personalized-fashion-recommendations'

# Load CLIP Model (MUST match local app: ViT-B-32 + laion2b_s34b_b79k)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model = model.to(device)
model.eval()
print('CLIP model loaded successfully!')

## 2. Process Dataset into Batches
We'll sample 5000 images for a quick test. Change `NUM_SAMPLES` to `None` for the full 105K dataset!

In [ ]:
articles_df = pd.read_csv(f'{DATA_DIR}/articles.csv')
print(f'Total articles in CSV: {len(articles_df):,}')

# ============================================================
# CHANGE THIS: Set to None for the full 105K product catalog
# ============================================================
NUM_SAMPLES = 5000

if NUM_SAMPLES:
    articles_df = articles_df.sample(n=NUM_SAMPLES, random_state=42)
    print(f'Sampled {NUM_SAMPLES:,} articles')

article_ids = []
images = []

print('Validating image paths...')
for idx, row in tqdm(articles_df.iterrows(), total=len(articles_df)):
    article_id = str(row['article_id']).zfill(10)
    folder = article_id[:3]
    path = f'{DATA_DIR}/images/{folder}/{article_id}.jpg'
    if os.path.exists(path):
        article_ids.append(article_id)
        images.append(path)

print(f'\nValid images found: {len(images):,} / {len(articles_df):,}')

## 3. Extract Embeddings using CLIP

In [ ]:
BATCH_SIZE = 256
all_embeddings = []

print(f'Running CLIP embeddings on {len(images):,} images...')
with torch.no_grad():
    for i in tqdm(range(0, len(images), BATCH_SIZE)):
        batch_paths = images[i:i + BATCH_SIZE]
        batch_images = []
        for path in batch_paths:
            img = Image.open(path).convert('RGB')
            batch_images.append(preprocess(img))
        
        image_tensor = torch.stack(batch_images).to(device)
        image_features = model.encode_image(image_tensor)
        # CRITICAL: Normalize for cosine similarity
        image_features /= image_features.norm(dim=-1, keepdim=True)
        all_embeddings.append(image_features.cpu().numpy())

embeddings = np.vstack(all_embeddings).astype('float32')
print(f'Embeddings Shape: {embeddings.shape}')

## 4. Build FAISS Index and Export
Download both files from the **Output** tab on the right panel after this cell runs.

In [ ]:
D = embeddings.shape[1]  # 512 dimensions
index = faiss.IndexFlatIP(D)  # Inner Product = Cosine Similarity for normalized vectors
index.add(embeddings)

print(f'FAISS Index built with {index.ntotal:,} vectors')

# Save Index
faiss.write_index(index, 'm2_clip_faiss.bin')

# Save mapping
pd.DataFrame({'article_id': article_ids}).to_csv('m2_faiss_mapping.csv', index=False)

# Show file sizes
faiss_size = os.path.getsize('m2_clip_faiss.bin') / (1024*1024)
mapping_size = os.path.getsize('m2_faiss_mapping.csv') / 1024
print(f'\nm2_clip_faiss.bin  : {faiss_size:.1f} MB')
print(f'm2_faiss_mapping.csv: {mapping_size:.1f} KB')
print('\nSuccessfully Exported! Download both files from the Output folder on the right panel.')